# Building the contrastive detector (SupCon)

**Part 1** assembles the supervised-contrastive detector: the SupCon loss, the encoder + projection head, the training loop, and a sweep over the projection dimension. **Part 2** is the systematic study — batch sizes 512 to 4096, with the linear evaluation done with and without the projection head. **Part 3** cleans the data first: malware samples that sit almost on top of benign ones are filtered out (the "denoising" step), and the detector is retrained on the cleaned data — this is where the `combined_model_denoised` weights used later come from.

**Where this fits:** this section tests a different defence idea — train the detector with **supervised contrastive learning** so its feature space separates the classes more cleanly, then attack the result with the machinery of `03-attacks/`. The CW machinery aimed at these models continues the lineage of `03-attacks/cw_sine_lineage.ipynb`.

*Consolidated from `contrast_learning.ipynb`, `contrast_learning2.ipynb`, `contrast_learning5.ipynb` and `denoising_dataset.ipynb` (originally in `mal_adv5` and `contrastive-learning`). Repeated train/evaluate cycles are shown once per configuration. Outputs cleared.*

## Part 1 — The pieces and the first builds

### Step 1 — Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader,WeightedRandomSampler
from sklearn.model_selection import train_test_split
from scipy import sparse
import gdown
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc
import time
import matplotlib.pyplot as plt
import os
import math

torch.manual_seed(0)
np.random.seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz'
]

output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()

# Combine features and labels into datasets
# we use dtype=torch.int8, for Memory-Efficient here, later we will convert to float
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

# Define the DataLoader for training, validation, and test sets
batch_size = 4096
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### Step 2 — The SupCon loss, the encoder, the training loop

In [ ]:
from sklearn.manifold import TSNE
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def extract_features(model, dataloader):
    model.eval()  # Set the model to evaluation mode
    features = []
    labels = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            outputs = model(inputs.to(torch.float32))
            features.append(outputs.cpu().numpy())
            labels.append(targets.cpu().numpy())

    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)
    return features, labels

def apply_tsne(features):
    tsne = TSNE(n_components=2, random_state=42)
    tsne_results = tsne.fit_transform(features)
    return tsne_results

def plot_tsne_and_density(tsne_results, labels):
    df = pd.DataFrame({
        'tsne_1': tsne_results[:, 0],
        'tsne_2': tsne_results[:, 1],
        'label': labels.squeeze()
    })

    # Create subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 14))

    # t-SNE scatter plot
    sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, ax=ax1, legend='full')
    ax1.set_title('t-SNE Scatter Plot')

    # Density plot
    #sns.kdeplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, fill=True, ax=ax2)
    # Density plot with transparency and contours
    sns.kdeplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, fill=True, alpha=0.70, ax=ax2)
    #sns.kdeplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, ax=ax2, linewidths=1)
    ax2.set_title('Density Plot')

    plt.show()

In [ ]:
class SupConLoss(nn.Module):
    """Supervised Contrastive Learning: https://arxiv.org/pdf/2004.11362.pdf.
    It also supports the unsupervised contrastive loss in SimCLR"""
    def __init__(self, temperature=0.07, contrast_mode='all',
                 base_temperature=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        """Compute loss for model. If both `labels` and `mask` are None,
        it degenerates to SimCLR unsupervised loss:
        https://arxiv.org/pdf/2002.05709.pdf

        Args:
            features: hidden vector of shape [bsz, n_views, ...].
            labels: ground truth of shape [bsz].
            mask: contrastive mask of shape [bsz, bsz], mask_{i,j}=1 if sample j
                has the same class as sample i. Can be asymmetric.
        Returns:
            A loss scalar.
        """
        device = (torch.device('cuda')
                  if features.is_cuda
                  else torch.device('cpu'))

        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             'at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        # compute logits
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        # for numerical stability
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # tile mask
        mask = mask.repeat(anchor_count, contrast_count)
        # mask-out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positive
        # modified to handle edge cases when there is no positive pair
        # for an anchor point.
        # Edge case e.g.:-
        # features of shape: [4,1,...]
        # labels:            [0,1,1,2]
        # loss before mean:  [nan, ..., ..., nan]
        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, 1, mask_pos_pairs)
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs

        # loss
        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss

In [ ]:
epochs = 200
lr_decay_rate = 0.1
learning_rate = 0.1
lrs = []
for epoch in range(epochs):
        #print('******',epoch)
        lr = learning_rate
        eta_min = lr * (lr_decay_rate ** 3)
        #print( math.cos(math.pi * epoch / epochs))
        lr = eta_min + (lr - eta_min) * (1 + math.cos(math.pi * epoch / epochs)) / 2
        #print(lr)
        lrs.append(lr)

plt.plot(lrs)
plt.show()

In [ ]:
import math
import numpy as np

def adjust_learning_rate(optimizer, epoch, epochs, learning_rate, lr_decay_rate, cosine, lr_decay_epochs=None):
    lr = learning_rate
    if cosine:
        eta_min = lr * (lr_decay_rate ** 3)
        lr = eta_min + (lr - eta_min) * (
                1 + math.cos(math.pi * epoch / epochs)) / 2
    else:
        if lr_decay_epochs is not None:
            steps = np.sum(epoch > np.asarray(lr_decay_epochs))
            if steps > 0:
                lr = lr * (lr_decay_rate ** steps)

    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

def warmup_learning_rate(epoch, batch_id, total_batches, optimizer, warm=False):
    # warm-up for large-batch training,
    if batch_size > 256:
        warm = True
    if warm:
        warmup_from = 0.01
        warm_epochs = 10
        if cosine:
            eta_min = learning_rate * (lr_decay_rate ** 3)
            warmup_to = eta_min + (learning_rate - eta_min) * (
                    1 + math.cos(math.pi * warm_epochs / epochs)) / 2
        else:
            warmup_to = learning_rate

        if epoch <= warm_epochs:
            p = (batch_id + (epoch - 1) * total_batches) / (warm_epochs * total_batches)
            lr = warmup_from + p * (warmup_to - warmup_from)

            for param_group in optimizer.param_groups:
                param_group['lr'] = lr


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def save_model(model, optimizer, epoch, save_file):
    print('==> Saving...')
    state = {
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'epoch': epoch,
    }
    torch.save(state, save_file)
    del state

In [ ]:
# Define the Encoder Network
class Encoder(nn.Module):
    def __init__(self, input_dim=10000, hidden_dim1=200, hidden_dim2=200, dropout_prob=0.6):
        super(Encoder, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        #self.dropout = nn.Dropout(dropout_prob)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        #x = self.dropout(x)
        x = self.fc2(x)
        out = self.relu(x)
        return out

# Define the SupConNet Network
class SupConNet(nn.Module):
    """Backbone + projection head"""
    def __init__(self, input_dim=10000, hidden_dim1=200, hidden_dim2=200, dropout_prob=0.6, feat_dim=128):
        super(SupConNet, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim1, hidden_dim2, dropout_prob)
        self.head = nn.Linear(hidden_dim2, feat_dim)

    def forward(self, x):
        feat = F.normalize(self.encoder(x), dim=1)
        feat = F.normalize(self.head(feat), dim=1)
        return feat

# Define the Linear Classifier
class LinearClassifier(nn.Module):
    """Linear classifier"""
    def __init__(self, feat_dim=200, num_classes=2):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(feat_dim, num_classes)

    def forward(self, features):
        return self.fc(features)

In [ ]:
def train(model, criterion, optimizer, train_loader, val_loader, epochs, device):
    for epoch in range(1, epochs + 1):
      adjust_learning_rate(optimizer, epoch, epochs, learning_rate, lr_decay_rate, cosine)

      model.train()
      losses = AverageMeter()
      for idx, (samples, labels) in enumerate(train_loader):
          samples, labels = samples.to(torch.float32).to(device), labels.to(device)

          bsz = labels.shape[0]

          # warm-up learning rate
          warmup_learning_rate(epoch, idx, len(train_loader), optimizer)

          # compute loss
          features = model(samples)
          #features.unsqueeze(1) to change from ([batch, 64]) to ([batch, 1, 64]), 1 for view
          loss = criterion(features.unsqueeze(1), labels)

          # update metric
          losses.update(loss.item(), bsz)

          # SGD
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

      # print info
      if (epoch) % 10 == 0:
          print('Train: [{0}]\t'
                'loss ({loss.avg:.3f})'.format(epoch, loss=losses))



      # save model
      if epoch % 100 == 0:
              save_file = os.path.join('ckpt_epoch_{epoch}.pth'.format(epoch=epoch))
              save_model(model, optimizer, epoch, save_file)

    # save the last model
    save_file = os.path.join('last.pth')
    save_model(model, optimizer, epochs, save_file)

### Step 3 — The projection-dimension sweep (64 / 128 / 32)

In [ ]:
model = SupConNet(feat_dim=64).to(device)
criterion = SupConLoss(temperature=0.07)

weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.005
momentum = 0.9
cosine = True
optimizer =  optim.SGD(model.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)
#optimizer = optim.Adam(model.parameters(), lr=learning_rate)
epochs = 500
train(model, criterion, optimizer, train_loader, val_loader, epochs, device)

In [ ]:
# Extract features from validation set
features, labels = extract_features(model.cpu(), val_loader)

# Apply t-SNE
tsne_results = apply_tsne(features)

# Plot joint plot
plot_tsne_and_density(tsne_results, labels)

In [ ]:
model = SupConNet(feat_dim=128).to(device)
criterion = SupConLoss(temperature=0.07)

weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.005
momentum = 0.9
cosine = True
optimizer =  optim.SGD(model.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)
#optimizer = optim.Adam(model.parameters(), lr=learning_rate)
epochs = 200
train(model, criterion, optimizer, train_loader, val_loader, epochs, device)

In [ ]:
model = SupConNet(feat_dim=32).to(device)
criterion = SupConLoss(temperature=0.07)

weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.005
momentum = 0.9
cosine = True
optimizer =  optim.SGD(model.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)
#optimizer = optim.Adam(model.parameters(), lr=learning_rate)
epochs = 200
train(model, criterion, optimizer, train_loader, val_loader, epochs, device)

In [ ]:
model = SupConNet(feat_dim=128).to(device)
criterion = SupConLoss(temperature=0.2)

weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.005
momentum = 0.9
cosine = True
optimizer =  optim.SGD(model.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)
#optimizer = optim.Adam(model.parameters(), lr=learning_rate)
epochs = 200
train(model, criterion, optimizer, train_loader, val_loader, epochs, device)

In [ ]:
model = SupConNet(feat_dim=128).to(device)
criterion = SupConLoss(temperature=0.04)

weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.005
momentum = 0.9
cosine = True
optimizer =  optim.SGD(model.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)
#optimizer = optim.Adam(model.parameters(), lr=learning_rate)
epochs = 200
train(model, criterion, optimizer, train_loader, val_loader, epochs, device)

In [ ]:
model = SupConNet(hidden_dim1=1024, hidden_dim2=256,feat_dim=128).to(device)
criterion = SupConLoss(temperature=0.07)

weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.005
momentum = 0.9
cosine = True
optimizer =  optim.SGD(model.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)
#optimizer = optim.Adam(model.parameters(), lr=learning_rate)
epochs = 200
train(model, criterion, optimizer, train_loader, val_loader, epochs, device)

In [ ]:
model.to(device)

In [ ]:
# training routine
epochs = 2000
for epoch in range(1, epochs + 1):
    adjust_learning_rate(optimizer, epoch, epochs, learning_rate, lr_decay_rate, cosine)

    model.train()
    losses = AverageMeter()
    for idx, (samples, labels) in enumerate(train_loader):
        samples, labels = samples.to(torch.float32).to(device), labels.to(device)

        bsz = labels.shape[0]

        # warm-up learning rate
        warmup_learning_rate(epoch, idx, len(train_loader), optimizer)

        # compute loss
        features = model(samples)
        #features.unsqueeze(1) to change from ([batch, 64]) to ([batch, 1, 64]), 1 for view
        loss = criterion(features.unsqueeze(1), labels)

        # update metric
        losses.update(loss.item(), bsz)

        # SGD
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # print info
        if (idx + 1) % 200 == 0:
            print('Train: [{0}][{1}/{2}]\t'
                  'loss {loss.val:.3f} ({loss.avg:.3f})'.format(epoch, idx + 1, len(train_loader), loss=losses))



    # save model
    print(losses.avg)
    if epoch % 50 == 0:
            save_file = os.path.join('ckpt_epoch_{epoch}.pth'.format(epoch=epoch))
            save_model(model, optimizer, epoch, save_file)

# save the last model
save_file = os.path.join('last.pth')
save_model(model, optimizer, epochs, save_file)

In [ ]:
# Extract features from validation set
features, labels = extract_features(model.cpu(), train_loader)

# Apply t-SNE
tsne_results = apply_tsne(features)

# Plot joint plot
plot_tsne_and_density(tsne_results, labels)

In [ ]:
model = SupConNet().to(device)
criterion = SupConLoss(temperature=0.07)

weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.005
momentum = 0.9
cosine = True
optimizer =  optim.SGD(model.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)

In [ ]:
def model_evaluation(model, test_loader, device):
    model.eval()  # Set the model to evaluation mode
    predictions = []
    true_labels = []

    with torch.no_grad():
        for X_batch, labels_batch in test_loader:
            X_batch, labels_batch = X_batch.to(torch.float32).to(device), labels_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.topk(outputs, k=1)
            predictions.extend(predicted.tolist())
            true_labels.extend(labels_batch.tolist())

    # Convert predictions and true labels to numpy arrays
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)

    # Calculate and print test accuracy
    accuracy = accuracy_score(true_labels, predictions)
    balanced_acc = balanced_accuracy_score(true_labels, predictions)
    print(f'Test Accuracy: {accuracy:.4f}')
    print(f'Test balanced Accuracy: {balanced_acc:.4f}')

    # Calculate and print precision, recall, and F1-score
    precision = precision_score(true_labels, predictions)
    recall = recall_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions)

    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-score: {f1:.4f}')

## Part 2 — The batch-size study

### Step 4 — The full training/evaluation harness

# Load dataset and libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader,WeightedRandomSampler
from sklearn.model_selection import train_test_split
from scipy import sparse
import gdown
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc
import time
import os
import math
from sklearn.manifold import TSNE
import seaborn as sns
import pandas as pd
import tqdm

torch.manual_seed(0)
np.random.seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()

# Combine features and labels into datasets
# we use dtype=torch.int8, for Memory-Efficient here, later we will convert to float
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

# SupContrast Model (Encoder + Projector)

In [ ]:
def extract_features(model, dataloader):
    model.cpu()
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            outputs = model(inputs.to(torch.float32))
            features.append(outputs.cpu().numpy())
            labels.append(targets.cpu().numpy())

    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)

    return features, labels


def apply_tsne(features):
    tsne = TSNE(n_components=2, random_state=42)
    tsne_results = tsne.fit_transform(features)
    return tsne_results

def plot_tsne_and_density(tsne_results, labels):
    df = pd.DataFrame({
        'tsne_1': tsne_results[:, 0],
        'tsne_2': tsne_results[:, 1],
        'label': labels.squeeze()
    })

    # Create subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 14))

    # t-SNE scatter plot
    sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, ax=ax1, legend='full')
    ax1.set_title('t-SNE Scatter Plot')

    # Density plot with transparency and contours
    sns.kdeplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, fill=True, alpha=0.70, ax=ax2)
    ax2.set_title('Density Plot')

    plt.show()

In [ ]:
# Define the Encoder Network
class Encoder(nn.Module):
    def __init__(self, input_dim=10000, hidden_dim1=200, hidden_dim2=200, dropout_prob=0.6):
        super(Encoder, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        #self.dropout = nn.Dropout(dropout_prob)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        #x = self.dropout(x)
        x = self.fc2(x)
        out = self.relu(x)
        return out

# Define the SupConNet Network
class SupConNet(nn.Module):
    """Backbone + projection head"""
    def __init__(self, input_dim=10000, hidden_dim1=200, hidden_dim2=200, dropout_prob=0.6, feat_dim=128):
        super(SupConNet, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim1, hidden_dim2, dropout_prob)
        self.head = nn.Linear(hidden_dim2, feat_dim)

    def forward(self, x):
        feat = self.encoder(x)
        feat = F.normalize(self.head(feat), dim=1)
        return feat

# Define the Linear Classifier
class LinearClassifier(nn.Module):
    """Linear classifier"""
    def __init__(self, feat_dim=200, num_classes=2):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(feat_dim, num_classes)

    def forward(self, features):
        return self.fc(features)

class CombinedModel(nn.Module):
    def __init__(self, encoder, classifier):
        super(CombinedModel, self).__init__()
        self.encoder = encoder
        self.classifier = classifier

    def forward(self, x):
        with torch.no_grad():  # Ensure encoder weights are not updated
            features = self.encoder(x)
        output = self.classifier(features)
        return output

In [ ]:
def adjust_learning_rate(optimizer, epoch, epochs, learning_rate, lr_decay_rate, cosine):
    lr = learning_rate
    if cosine:
        eta_min = lr * (lr_decay_rate ** 3)
        lr = eta_min + (lr - eta_min) * (1 + math.cos(math.pi * epoch / epochs)) / 2
    else:
        steps = np.sum(epoch > np.asarray([int(0.7*epochs), int(0.8*epochs), int(0.9*epochs)]))  # Update as per your schedule
        if steps > 0:
            lr = lr * (lr_decay_rate ** steps)

    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

def warmup_learning_rate(batch_size, epoch, batch_id, total_batches, optimizer, cosine, learning_rate, lr_decay_rate, warm_up=False):
    if (warm_up) or (batch_size > 256):
        warm_epochs = 10
        warmup_from = (0.1 * learning_rate)

        if epoch <= warm_epochs:
            p = (batch_id + (epoch - 1) * total_batches) / (warm_epochs * total_batches)
            if cosine:
                eta_min = learning_rate * (lr_decay_rate ** 3)
                warmup_to = eta_min + (learning_rate - eta_min) * (1 + math.cos(math.pi * warm_epochs / epochs)) / 2
            else:
                warmup_to = learning_rate

            lr = warmup_from + p * (warmup_to - warmup_from)
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr

def save_model(model, optimizer, epoch, save_file):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, save_file)

In [ ]:
model = SupConNet(feat_dim=128).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.01
cosine = True
batch_size = 1024
lrs = []
epochs = 200
for epoch in range(1, epochs + 1):
        adjust_learning_rate(optimizer, epoch, epochs, learning_rate, lr_decay_rate, cosine)

        for idx, _ in enumerate(train_loader):

            # Warm-up learning rate
            warmup_learning_rate(batch_size, epoch, idx, len(train_loader), optimizer, cosine, learning_rate, lr_decay_rate, warm_up=False)

            for param_group in optimizer.param_groups:
                lrs.append(param_group['lr'])


plt.plot(lrs)
plt.show()

In [ ]:
def train(model, criterion, optimizer, train_loader, val_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device, phase2 = False):

    for epoch in range(1, epochs + 1):
        adjust_learning_rate(optimizer, epoch, epochs, learning_rate, lr_decay_rate, cosine)

        model.train()
        epoch_loss = 0
        correct = 0
        total_samples = 0

        for idx, (samples, labels) in enumerate(train_loader):

            bsz = labels.shape[0]

            if ~phase2 :
                # avoid small batch_size
                if bsz < batch_size:
                  continue

            samples, labels = samples.to(torch.float32).to(device), labels.to(device)
            bsz = labels.shape[0]

            # Warm-up learning rate
            warmup_learning_rate(batch_size, epoch, idx, len(train_loader), optimizer, cosine, learning_rate, lr_decay_rate, warm_up=False)

            # Compute loss
            outputs = model(samples)
            if phase2 :
                loss = criterion(outputs, labels.view(-1).long())
            else:
                loss = criterion(outputs.unsqueeze(1), labels)

            # SGD
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Update total loss and samples count
            epoch_loss += loss.item() * bsz
            total_samples += bsz

            if phase2 :
                pred = outputs.argmax(dim=1, keepdim=True)
                correct += pred.eq(labels.view_as(pred)).sum().item()


        # Calculate average loss for the epoch
        avg_loss = epoch_loss / total_samples
        if phase2 :
            accuracy = 100.0 * correct / total_samples


        # Print info
        if epoch % 5 == 0:
            if phase2 :
                val_loss, val_acc = validate(model, classifier, val_loader, criterion, device)
                print(f'Train: [{epoch}]\tLoss: {avg_loss:.3f}\tAccuracy: {accuracy:.2f}%\tValidation: [{epoch}]\tLoss: {val_loss:.3f}\tAccuracy: {val_acc:.2f}%')
            else:
                print(f'Train: [{epoch}]\tLoss: {avg_loss:.3f}')

        # Save model
        if epoch % 100 == 0:
            save_file = os.path.join(f'ckpt_epoch_{epoch}.pth')
            save_model(model, optimizer, epoch, save_file)

In [ ]:
def validate(model, classifier, val_loader, criterion, device):
    #model.to(device)
    model.eval()
    val_loss = 0
    correct = 0
    total_samples = 0

    with torch.no_grad():
        for samples, labels in val_loader:
            samples, labels = samples.to(torch.float32).to(device), labels.to(device)
            outputs = model(samples)

            loss = criterion(outputs, labels.view(-1).long())

            val_loss += loss.item() * samples.size(0)
            pred = outputs.argmax(dim=1, keepdim=True)
            correct += pred.eq(labels.view_as(pred)).sum().item()
            total_samples += labels.size(0)

    avg_loss = val_loss / total_samples
    accuracy = 100.0 * correct / total_samples

    return avg_loss, accuracy

### Step 5 — Batch size 1024, and the linear evaluation (with and without the projection head)

# batch size 1024

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 1024
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
model_contrast = SupConNet(feat_dim=128).to(device)
criterion = SupConLoss(temperature=0.07)
weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.05
momentum = 0.9
cosine = True
optimizer =  optim.SGD(model_contrast.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)
#optimizer = optim.Adam(model_contrast.parameters(), lr=learning_rate)
epochs = 200
train(model_contrast, criterion, optimizer, train_loader, val_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device, phase2 = False)

In [ ]:
# Extract the pretrained encoder
pretrained_encoder = model_contrast.encoder

# Set the parameters of the encoder to not require gradients
for param in pretrained_encoder.parameters():
    param.requires_grad = False

In [ ]:
classifier2 = LinearClassifier().to(device)
combined_model2 = CombinedModel(pretrained_encoder, classifier2).to(device)

class_weights = torch.tensor([0.11, 0.89]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(combined_model2.classifier.parameters(), lr=learning_rate)

learning_rate = 0.005
cosine = True
epochs = 50

train(combined_model2, criterion, optimizer, train_loader, val_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device, phase2 = True)

In [ ]:
loaders = {'Train': train_loader, 'Validation': val_loader, 'Test': test_loader}

for loader_name, loader in loaders.items():
    print(f'\n ******** {loader_name} *********')
    model_evaluation(combined_model2, loader, device)

In [ ]:
# with projection
classifier = LinearClassifier(feat_dim=128, num_classes=2).to(device)
combined_model = CombinedModel(model_contrast, classifier).to(device)

class_weights = torch.tensor([0.11, 0.89]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(combined_model.classifier.parameters(), lr=learning_rate)

learning_rate = 0.005
cosine = True
epochs = 30

train(combined_model, criterion, optimizer, train_loader, val_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device, phase2 = True)

In [ ]:
loaders = {'Train': train_loader, 'Validation': val_loader, 'Test': test_loader}

for loader_name, loader in loaders.items():
    print(f'\n ******** {loader_name} *********')
    model_evaluation(combined_model, loader, device)

In [ ]:
# Extract features from validation set
features, labels = extract_features(model_contrast.cpu(), val_loader)

# Apply t-SNE
tsne_results = apply_tsne(features)

# Plot joint plot
plot_tsne_and_density(tsne_results, labels)

### Step 6 — Batch size 512 (two learning rates)

# batch size 512

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 512
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
model = SupConNet(feat_dim=128).to(device)
criterion = SupConLoss(temperature=0.07)
weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.1
momentum = 0.9
cosine = True
optimizer =  optim.SGD(model.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)
#optimizer = optim.Adam(model.parameters(), lr=learning_rate)
epochs = 110
train_step1(model, criterion, optimizer, train_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device)

In [ ]:
learning_rate = 0.01
cosine = True

classifier = LinearClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(), lr=learning_rate)
epochs = 50
train_step2(model, classifier, criterion, optimizer, train_loader, val_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device)

In [ ]:
loaders = {'Train': train_loader, 'Validation': val_loader, 'Test': test_loader}

for loader_name, loader in loaders.items():
    print(f'\n ******** {loader_name} *********')
    model_evaluation(model, classifier, loader, device)

In [ ]:
learning_rate = 0.001
cosine = True

classifier = LinearClassifier().to(device)
class_weights = torch.tensor([0.11, 0.89]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(classifier.parameters(), lr=learning_rate)
epochs = 50
train_step2(model, classifier, criterion, optimizer, train_loader, val_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device)

In [ ]:
# Extract features from validation set
features, labels = extract_features(model.encoder.cpu(), val_loader)

# Apply t-SNE
tsne_results = apply_tsne(features)

# Plot joint plot
plot_tsne_and_density(tsne_results, labels)

### Step 7 — Batch size 2048

# batch size 2048

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 2048
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
learning_rate = 0.05
cosine = True

classifier = LinearClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(), lr=learning_rate)
epochs = 50
train_step2(model, classifier, criterion, optimizer, train_loader, val_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device)

### Step 8 — Batch size 4096

# batch size 4096

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 4096
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
model = SupConNet(feat_dim=128).to(device)
criterion = SupConLoss(temperature=0.07)
weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.2
momentum = 0.9
cosine = True
optimizer =  optim.SGD(model.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)
#optimizer = optim.Adam(model.parameters(), lr=learning_rate)
epochs = 110
train_step1(model, criterion, optimizer, train_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device)

In [ ]:
learning_rate = 0.1
cosine = True

classifier = LinearClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(), lr=learning_rate)
epochs = 50
train_step2(model, classifier, criterion, optimizer, train_loader, val_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device)

## Part 3 — Denoising the dataset

### Step 9 — Finding the malware samples that sit on top of benign ones

In [ ]:
# Combine the tensors along the batch dimension (dimension 0)
combined_x = torch.cat((X_train, X_val, X_test), dim=0)
combined_y = torch.cat((y_train, y_val, y_test), dim=0)

print(combined_x.size())
print(combined_y.size())

In [ ]:
benigns = (combined_x[combined_y.squeeze()==0]).to(device)
malwares = (combined_x[combined_y.squeeze()==1]).to(device)
print(benigns.shape)
print(malwares.shape)

In [ ]:
similar = []
for i,mal in enumerate(malwares):
  differences = (torch.abs(benigns - mal)).sum(dim=1)
  value = torch.min(differences)
  if value < 5:
      print(value)
      print(i)
      similar.append(i)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
import torch

# Assuming malwares and benigns are your input tensors
# malwares = torch.load('path_to_malwares_tensor')
# benigns = torch.load('path_to_benigns_tensor')

# For demonstration, let's assume malwares and benigns are already loaded
# malwares has size [1083, 10000]
# benigns has size [8479, 10000]

# Define the batch size for processing
batch_size = 2 # Adjust based on your memory constraints

# Store the indices of the closest benign samples for each malware sample
closest_values = (torch.empty(malwares.size(0), dtype=torch.long)).to(device)

# Process in batches
for i in range(0, malwares.size(0), batch_size):
    # Select the current batch
    batch_malwares = malwares[i:i+batch_size]

    # Expand the current batch and benigns for broadcasting
    batch_malwares_expanded = batch_malwares.unsqueeze(1)  # shape [batch_size, 1, 10000]
    benigns_expanded = benigns.unsqueeze(0)  # shape [1, 8479, 10000]

    # Compute the difference
    differences = torch.abs(batch_malwares_expanded - benigns_expanded)  # shape [batch_size, 8479, 10000]

    # Compute the Euclidean distance (L2 norm) for each pair
    distances = torch.sum(differences, dim=2)  # shape [batch_size, 8479]


    # Find the index of the minimum distance in benigns for each sample in the batch
    values, _ = torch.min(distances, dim=1)  # shape [batch_size]

    # Store the results
    closest_values[i:i+batch_size] = values

    if i %500==0:
      print(i)

In [ ]:
(closest_values<5).sum()

In [ ]:
closest_values[closest_values<5]

In [ ]:
malwares = malwares[closest_values>4]
print(malwares.shape)

In [ ]:
print(benigns.shape)

In [ ]:
import torch
import numpy as np

def split_data_without_shuffle(tensor, train_ratio, val_ratio, test_ratio):
    num_samples = tensor.size(0)

    train_end = int(train_ratio * num_samples)
    val_end = train_end + int(val_ratio * num_samples)

    train_data = tensor[:train_end]
    val_data = tensor[train_end:val_end]
    test_data = tensor[val_end:]

    return train_data, val_data, test_data


# Ratios for splitting
train_ratio = 0.6
val_ratio = 0.2
test_ratio = 0.2

# Split malware data without shuffling
malware_train, malware_val, malware_test = split_data_without_shuffle(malwares, train_ratio, val_ratio, test_ratio)

# Split benign data without shuffling
benign_train, benign_val, benign_test = split_data_without_shuffle(benigns, train_ratio, val_ratio, test_ratio)

# Combine malware and benign data for each set
X_train = torch.cat((malware_train, benign_train), dim=0)
y_train = torch.cat((torch.ones(malware_train.size(0), 1, dtype=torch.int8), torch.zeros(benign_train.size(0), 1, dtype=torch.int8)), dim=0)

X_val = torch.cat((malware_val, benign_val), dim=0)
y_val = torch.cat((torch.ones(malware_val.size(0), 1, dtype=torch.int8), torch.zeros(benign_val.size(0), 1, dtype=torch.int8)), dim=0)

X_test = torch.cat((malware_test, benign_test), dim=0)
y_test = torch.cat((torch.ones(malware_test.size(0), 1, dtype=torch.int8), torch.zeros(benign_test.size(0), 1, dtype=torch.int8)), dim=0)

print("Train shapes: ", X_train.shape, y_train.shape)
print("Validation shapes: ", X_val.shape, y_val.shape)
print("Test shapes: ", X_test.shape, y_test.shape)

### Step 10 — Retraining on the denoised data, and saving the model

In [ ]:
X_test.dtype

In [ ]:
# Convert tensors to numpy arrays
X_train_np = X_train.cpu().numpy()
X_val_np = X_val.cpu().numpy()
X_test_np = X_test.cpu().numpy()
y_train_np = y_train.cpu().numpy()
y_val_np = y_val.cpu().numpy()
y_test_np = y_test.cpu().numpy()

# Save tensors using numpy.savez_compressed
np.savez_compressed('train_data.npz', X_train=X_train_np, y_train=y_train_np)
np.savez_compressed('val_data.npz', X_val=X_val_np, y_val=y_val_np)
np.savez_compressed('test_data.npz', X_test=X_test_np, y_test=y_test_np)

In [ ]:
# Load tensors
loaded_train_data = np.load('train_data.npz')
loaded_val_data = np.load('val_data.npz')
loaded_test_data = np.load('test_data.npz')

loaded_X_train_np = loaded_train_data['X_train']
loaded_y_train_np = loaded_train_data['y_train']
loaded_X_val_np = loaded_val_data['X_val']
loaded_y_val_np = loaded_val_data['y_val']
loaded_X_test_np = loaded_test_data['X_test']
loaded_y_test_np = loaded_test_data['y_test']

# Convert numpy arrays back to tensors
X_train = torch.tensor(loaded_X_train_np, dtype=torch.int8)
y_train = torch.tensor(loaded_y_train_np, dtype=torch.int8)
X_val = torch.tensor(loaded_X_val_np, dtype=torch.int8)
y_val = torch.tensor(loaded_y_val_np, dtype=torch.int8)
X_test = torch.tensor(loaded_X_test_np, dtype=torch.int8)
y_test = torch.tensor(loaded_y_test_np, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()

# Combine features and labels into datasets
# we use dtype=torch.int8, for Memory-Efficient here, later we will convert to float
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

print(n_mal/(n_mal+n_ben))

In [ ]:
for X_batch, y_batch in train_loader:
  print(X_batch[y_batch.squeeze()==0].shape)
  print(X_batch[y_batch.squeeze()==1].shape)
  break

# batch size 1024

In [ ]:
model_contrast = SupConNet(feat_dim=128).to(device)
criterion = SupConLoss(temperature=0.07)
weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.05
momentum = 0.9
cosine = True
#optimizer =  optim.SGD(model_contrast.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)
optimizer = optim.Adam(model_contrast.parameters(), lr=learning_rate)
epochs = 110
train(model_contrast, criterion, optimizer, train_loader, val_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device, phase2 = False)

In [ ]:
# Extract features from validation set
features, labels = extract_features(model_contrast.encoder.cpu(), val_loader)

# Apply t-SNE
tsne_results = apply_tsne(features)

# Plot joint plot
plot_tsne_and_density(tsne_results, labels)

In [ ]:
classifier = LinearClassifier().to(device)
combined_model = CombinedModel(pretrained_encoder, classifier).to(device)

class_weights = torch.tensor([0.1, 0.9]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(combined_model.classifier.parameters(), lr=learning_rate)

learning_rate = 0.001
cosine = True
epochs = 50

train(combined_model, criterion, optimizer, train_loader, test_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device, phase2 = True)

In [ ]:
# with projection head
# Extract features from validation set
features, labels = extract_features(combined_model.cpu(), val_loader)

# Apply t-SNE
tsne_results = apply_tsne(features)

# Plot joint plot
plot_tsne_and_density(tsne_results, labels)

In [ ]:
torch.save(combined_model, 'combined_model_denoised_adam.pth')

In [ ]:
model_contrast = SupConNet(feat_dim=128).to(device)
criterion = SupConLoss(temperature=0.07)
weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.05
momentum = 0.9
cosine = True
optimizer =  optim.SGD(model_contrast.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)
#optimizer = optim.Adam(model_contrast.parameters(), lr=learning_rate)
epochs = 110
train(model_contrast, criterion, optimizer, train_loader, val_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device, phase2 = False)

In [ ]:
classifier = LinearClassifier().to(device)
combined_model = CombinedModel(pretrained_encoder, classifier).to(device)

class_weights = torch.tensor([0.1, 0.9]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(combined_model.classifier.parameters(), lr=learning_rate)

learning_rate = 0.01
cosine = True
epochs = 50

train(combined_model, criterion, optimizer, train_loader, val_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device, phase2 = True)

In [ ]:
torch.save(combined_model, 'combined_model_denoised.pth')

In [ ]:
classifier = LinearClassifier().to(device)
combined_model2 = CombinedModel(pretrained_encoder, classifier).to(device)

class_weights = torch.tensor([0.1, 0.9]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

learning_rate = 0.05
cosine = True
epochs = 50
optimizer = optim.SGD(combined_model2.classifier.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)

train(combined_model2, criterion, optimizer, train_loader, val_loader, epochs, batch_size, learning_rate, lr_decay_rate, cosine, device, phase2 = True)

In [ ]:
# with projection head
# Extract features from validation set
features, labels = extract_features(combined_model2.cpu(), val_loader)

# Apply t-SNE
tsne_results = apply_tsne(features)

# Plot joint plot
plot_tsne_and_density(tsne_results, labels)